In [ ]:
import requests
import pandas as pd
import os

def fetch_aims_data(api_key, doi, site_name=None, size=1000):
    """
    Pulls data from the AIMS Data Platform API using a specific DOI.
    """
    base_url = f"https://api.aims.gov.au/data/v1.0/{doi}/data"
    headers = {"x-api-key": api_key}
    
    # Parameters for the API call
    params = {
        "size": size,
    }
    if site_name:
        params["site-name"] = site_name

    print(f"Requesting data for DOI: {doi}...")
    response = requests.get(base_url, headers=headers, params=params)
    
    if response.status_code == 200:
        data = response.json()
        # The API returns a list of records under the 'data' key
        df = pd.DataFrame(data['data'])
        return df
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None

# --- CONFIGURATION ---
API_KEY = "ae3OPUMdYs1kFhn0NkZTd53jNAowaO7TqH3qgvF1"
# DOI for LTMP Manta Tow Data (Example DOI, verify in AIMS Catalogue)
LTMP_DOI = "10.25845/5c09bf93f315d"

# 1. Fetch the data
df_aims = fetch_aims_data(API_KEY, LTMP_DOI, size=5000)

if df_aims is not None:
    # 2. Format for RNN: Convert 'time' to datetime and sort
    # AIMS usually returns 'time' or 'sample_date'
    date_col = 'time' if 'time' in df_aims.columns else 'sample_date'
    df_aims[date_col] = pd.to_datetime(df_aims[date_col])
    df_aims = df_aims.sort_values(by=[date_col])

    # 3. Save in an "Appropriate Format"
    # Parquet is superior for ML because it preserves data types (float/int/date)
    # whereas CSV turns everything back into strings.
    df_aims.to_parquet("aims_ltmp_ground_truth.parquet", index=False)
    df_aims.to_csv("aims_ltmp_ground_truth.csv", index=False)
    
    print("Success! Data saved as 'aims_ltmp_ground_truth.parquet'.")
    print(df_aims.head())

Requesting data for DOI: 10.25845/5c09bf93f315d...
Error 502: {"message": "Internal server error"}
